# 🍎 Pixels-to-Macros — Food Segmentation Training

**Before running:**  
1. Runtime → Change runtime type → **GPU** (T4 is free, A100 is faster on Colab Pro)  
2. Connect to a runtime, then run cells top-to-bottom.

Outputs (checkpoints + metrics) are saved to **Google Drive** so they survive session resets.

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Google Drive (saves checkpoints across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
OUTPUT_DIR = pathlib.Path('/content/drive/MyDrive/pixels-to-macros-training')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## Cell 3 — Clone repo

In [ ]:

import pathlib, subprocess

%cd /content

repo = pathlib.Path('/content/pixels-to-macros')

# Remove broken directory (exists but no .git)
if repo.exists() and not (repo / '.git').exists():
    print('Removing broken directory...')
    subprocess.run(['rm', '-rf', str(repo)], check=True)

if not repo.exists():
    !git clone https://github.com/JormayBusso/pixels-to-macros.git /content/pixels-to-macros

%cd /content/pixels-to-macros

# Force-sync to latest remote (fixes stale .py / __pycache__ issues)
!git fetch --all
!git reset --hard origin/dev

# Clear any cached .pyc that could shadow updated source files
!rm -rf training/__pycache__

print('\n=== training/ ===')
!ls training/


## Cell 4 — Install requirements

> This installs a CUDA-enabled PyTorch first, then the rest of `training/requirements.txt`.
> `coremltools` is excluded (CoreML export is macOS-only; do that step on your Mac later).

In [ ]:

# Install segmentation-models-pytorch and other training deps.
# We keep Colab's pre-installed torch (2.10+) and skip coremltools (macOS-only).
!grep -vE '^torch|^torchvision|coremltools' /content/pixels-to-macros/training/requirements.txt > /tmp/colab_reqs.txt
!pip install -q segmentation-models-pytorch timm
!pip install -q -r /tmp/colab_reqs.txt

import torch, numpy as np
print(f'torch={torch.__version__}  numpy={np.__version__}  CUDA={torch.cuda.is_available()}')


## Cell 5 — (Optional) Install SegFormer support

Only needed if you plan to train with `--model segformer`. Skip otherwise.

In [ ]:
# Uncomment to install:
# !pip install -q transformers
print('Skipped (remove comment above to install SegFormer support)')

## Cell 6 — Download FoodSeg103 dataset (~1.3 GB)

Skipped automatically if the dataset already exists.

In [ ]:

%cd /content/pixels-to-macros

import pathlib
train_dir = pathlib.Path('data/FoodSeg103/Images/img_dir/train')

if train_dir.exists() and len(list(train_dir.glob('*.jpg'))) > 100:
    print(f'Dataset already present ({len(list(train_dir.glob("*.jpg")))} train images). Skipping download.')
else:
    print('Downloading FoodSeg103 via HuggingFace (no quota limits)...')
    !pip install -q datasets
    !python scripts/download_hf_foodseg103.py
    count = len(list(train_dir.glob('*.jpg'))) if train_dir.exists() else 0
    print(f'\nDone! {count} train images ready.')


## Cell 7 — Training config

Adjust these variables before starting training.

In [ ]:

# ── Training config ───────────────────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 4              # reduce to 2 if OOM
IMG_SIZE      = 640
LR            = 2e-4
NUM_WORKERS   = 2
DATA_DIR      = './data/FoodSeg103'
OUTPUT_DIR    = '/content/drive/MyDrive/pixels-to-macros-training'

print(f'Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}')
print(f'Output: {OUTPUT_DIR}')


## Cell 8 — Run training 🚀

Training auto-resumes from the last checkpoint if one exists in `OUTPUT_DIR`.  
Progress bars appear in real time below this cell.

In [ ]:

%cd /content/pixels-to-macros

# Mount Drive (no-op if already mounted)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ckpt = pathlib.Path(OUTPUT_DIR) / 'last_checkpoint.pth'
if ckpt.exists():
    print(f'Resuming from checkpoint: {ckpt.stat().st_size / 1e6:.1f} MB')
else:
    print('No checkpoint found — starting fresh.')

print(f'\nEpochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR} | Output: {OUTPUT_DIR}\n')

!python training/train.py \
  --data_dir {DATA_DIR} \
  --output_dir {OUTPUT_DIR} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --img_size {IMG_SIZE} \
  --lr {LR} \
  --num_workers {NUM_WORKERS}


## Cell 9 — Live metrics watcher

Run this in a **separate cell** while Cell 8 is running to watch epoch-level results update every 10 seconds.  
Stop it with the ■ button or `Kernel → Interrupt`.

In [ ]:

import time, json, pathlib
from IPython.display import clear_output

metrics_path = pathlib.Path(OUTPUT_DIR) / 'metrics.json'
ckpt_path    = pathlib.Path(OUTPUT_DIR) / 'last_checkpoint.pth'
best_path    = pathlib.Path(OUTPUT_DIR) / 'best.pth'

def fmt(val, width, decimals=4):
    try:    return f'{float(val):>{width}.{decimals}f}'
    except: return f'{"—":>{width}}'

print(f'Watching {metrics_path}  (updates every 10 s) — interrupt to stop\n')

while True:
    clear_output(wait=True)
    print(f'Watching {metrics_path}\n')

    if metrics_path.exists():
        try:
            data = json.loads(metrics_path.read_text())
            print(f"{'Ep':>4}  {'TrainLoss':>9}  {'ValLoss':>8}  {'mIoU':>7}  {'PixAcc':>7}  {'BestIoU':>8}  {'LR':>8}  {'Time':>6}")
            print('─' * 68)
            for r in data[-15:]:
                val_ran = r.get('val_ran', True)
                val_col = fmt(r.get('val_loss'), 8) if val_ran else f'{"skipped":>8}'
                print(
                    f"{str(r.get('epoch', '?')):>4}  "
                    f"{fmt(r.get('train_loss'), 9)}  "
                    f"{val_col}  "
                    f"{fmt(r.get('val_miou'), 7)}  "
                    f"{fmt(r.get('pixel_acc'), 7)}  "
                    f"{fmt(r.get('best_miou'), 8)}  "
                    f"{fmt(r.get('lr'), 8, 6)}  "
                    f"{fmt(r.get('time_s'), 5, 0)}s"
                )
            valid_miou = [r['val_miou'] for r in data if r.get('val_miou') is not None]
            if valid_miou:
                print(f'\nBest mIoU: {max(valid_miou):.4f}  |  Epochs logged: {len(data)}')
            else:
                print(f'\nEpochs logged: {len(data)} — validation not yet run')
        except Exception as e:
            print(f'(waiting for metrics: {e})')
    else:
        print('No metrics.json yet — training may still be in the first epoch.')

    ckpt_size = f'{ckpt_path.stat().st_size / 1e6:.1f} MB' if ckpt_path.exists() else 'not saved yet'
    best_size = f'{best_path.stat().st_size / 1e6:.1f} MB' if best_path.exists() else 'not saved yet'
    print(f'\nlast_checkpoint.pth : {ckpt_size}')
    print(f'best.pth            : {best_size}')

    time.sleep(10)


## Cell 10 — Verify saved files on Drive

In [ ]:
import pathlib, json

out = pathlib.Path(OUTPUT_DIR)
files = sorted(out.glob('*'))

print(f'Files in {out}:')
for f in files:
    size = f.stat().st_size / 1e6
    print(f'  {f.name:<35} {size:>8.2f} MB')

metrics_file = out / 'metrics.json'
if metrics_file.exists():
    data = json.loads(metrics_file.read_text())
    print(f'\nTotal epochs logged: {len(data)}')
    if data:
        best = max(data, key=lambda r: r.get('val_miou', 0))
        print(f'Best epoch: {best["epoch"]}  mIoU: {best.get("val_miou",0):.4f}  pixel_acc: {best.get("val_pixel_acc",0):.4f}')

## Tips

| Situation | Fix |
|---|---|
| OOM (out of memory) | Lower `BATCH_SIZE` to 2, or `IMG_SIZE` to 512 |
| Session reset | Re-run cells 2–3, then Cell 8 — training auto-resumes from the last checkpoint |
| Faster training | Set `MODEL = 'resnet101'` (stronger backbone); enable `COMPILE = True` on PyTorch 2+ |
| SegFormer | Run Cell 5, then set `MODEL = 'segformer'` |
| Export CoreML | After training, copy `best.pth` to your Mac and run `python training/export_coreml.py --checkpoint training/output/best.pth` |